<a href="https://colab.research.google.com/github/adi-bhagwat-v-s/Makemore/blob/main/Makemore_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
import random

In [2]:
from google.colab import files
uploaded = files.upload()

Saving names.txt to names.txt


In [3]:
# read all the words
words = open('names.txt', 'r').read().split()

In [4]:
# Create a mapping between words and integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [5]:
def build_dataset(words):
  block_size = 3 # Context length - the number of characters that is used to predict the next one.
  X, Y = [], []
  for w in words:
    context = [0]*block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

''' training split, dev/validation split, test split - the entire data is divided into these splits.
This is usually done in the ratio of 8:1:1(80%, 10% and 10%)'''
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [6]:
g = torch.Generator().manual_seed(2147483647) # for reproducibilty
C = torch.randn((27, 10), generator=g) # Lookup table
# Hidden Layer
W1 = torch.randn((30, 200), generator=g) # 30 because there are three inputs each having a two diamensional embeddings(3*10), number of neuron is set to 100(random).
b1 = torch.rand(200, generator=g) # Biases - will be the same as the number of neurons
# Output Layer
W2 = torch.randn((200, 27), generator=g) # 100 because there are 100 neurons in the hidden layer and all the neurons in the output layer are fully connected.
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [7]:
for p in parameters:
  p.requires_grad = True

In [8]:
lre = torch.linspace(-3, 0, 100) # Learning rate exponential
lrs = 10**lre # lrs will be an array starting from -0.001 and increasing exponentialy to 1

In [9]:
for i in range(200000):
  # Minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,)) # used to speed up the training process. Plucks out 32 embeddings in an iteration.
  # Forward Pass
  emb = C[Xtr[ix]]
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
  ''' -1 lets pytorch guess what the number in the index position be, here -1 will be interpreted as the number of embeddingst.
  emb.view sqashes the [num_samples,3,2] emb array into [num_samples, 30] array so that dot product can be done with W1 which is a [30, 100] array'''
  logits = h @ W2 + b2 # logits or log digits are unnormalized scores predicted by the output layer, they are further normalized using an activation function.
  loss = F.cross_entropy(logits, Ytr[ix]) # Works the same as finding counts -> prob -> loss(log + mean)

  # Backward Pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # Update
  lr = 0.01 if i>=100000 else 0.1 # learning rate
  for p in parameters:
    p.data += -lr*p.grad

print("loss => ", loss.item())

loss =>  2.0645735263824463


In [10]:
# dev loss
ix = torch.randint(0, Xdev.shape[0], (32,))
emb = C[Xdev[ix]]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev[ix])
print("loss => ", loss)

loss =>  tensor(1.9138, grad_fn=<NllLossBackward0>)


In [11]:
# training loss
ix = torch.randint(0, Xtr.shape[0], (32,))
emb = C[Xtr[ix]]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr[ix])
print("loss => ", loss)

loss =>  tensor(1.9025, grad_fn=<NllLossBackward0>)


In [13]:
# Sample from the model
g = torch.Generator().manual_seed(2147483657)
for _ in range(10):
  out = []
  context = [0]*3 # 3 is the block size
  while True:
    emb = C[torch.tensor([context])]
    h = torch.tanh(emb.view(1, -1) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix==0:
      break

  print(''.join(itos[i] for i in out))

carmahzamille.
khy.
mili.
taty.
halayane.
mahnen.
deliah.
jareeix.
ramara.
chaiiv.
